# Chapter 7 lab — What should happen to missed scheduled runs?

Draft educator companion v1 · 8 September 2026 · 90 minutes.

Read [the chapter](https://www.profrod.ai/book/ch07-scheduling) alongside this lab. **Prerequisites:** Chapter 6 durable intake; integer division and explicit observed time.

You will build one explicitly scoped decision function, challenge it with independently authored cases, and trace the same concern through the cumulative runtime. The manuscript is where you build the full components; this notebook is a focused companion, not a claim that importing a runtime teaches its construction.

**Before running:** write your prediction. Keep the worked solution below closed until you have attempted the function. Download/open this notebook in an existing Jupyter environment using the book’s Python 3.14 interpreter after completing repository setup in the book conventions. Unlike the two Chapter 1 notebooks, this lab requires the local checkout and its locked book dependencies. It does not install packages, launch a hosted notebook, or require model credentials.

**Without a notebook server:** read and edit the cells in your editor, then run `uv run --python 3.14 python book/always_on/educator/run_lesson_v1.py --chapter 7 --output /tmp/lucy-ch07-class.json` from the repository root. Use a new output filename on each retained run. The runner executes the saved notebook and records student results separately from the worked example.


## 1. Predict (10 minutes)

A job is due at 100, repeats every ten seconds, and is observed at 139. How many work records should a coalescing policy create, how many runs were skipped, and when is the next due time?

Write both the expected result and the evidence that could disprove your explanation.


In [ ]:
import copy
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

if sys.version_info < (3, 14):
    raise RuntimeError("Use the book Python 3.14 environment for Chapters 2–16.")
# Open the notebook inside your source checkout, or set this path explicitly.
start = Path(os.environ.get("SOVEREIGN_AGENT_REPO", Path.cwd())).resolve()
ROOT = next(
    (p for p in (start, *start.parents) if (p / "book/always_on/checkpoints/ch07.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Set SOVEREIGN_AGENT_REPO to the Sovereign Agent checkout.")
CHECKPOINT = ROOT / "book/always_on/checkpoints/ch07.py"
EXPECTED_CHECKPOINT_SHA256 = "652947c8165a1810fea7860223e7372b83956af1e0fa5a2b9427d4c67d5147f9"
if hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest() != EXPECTED_CHECKPOINT_SHA256:
    raise RuntimeError("Checkpoint version differs from this lesson; use its matching release.")
print("Chapter 7 checkpoint bytes match this lesson. No model or channel has been called.")

## 2. Build your decision (25 minutes)

Implement decide(case) for integer due, now and positive interval. If paused or now < due, return [0,0,due]. Otherwise return [1, skipped, next_due] where skipped=(now-due)//interval and next_due=due+(skipped+1)*interval. One work item represents the latest coalesced wake. No wall-clock reads inside the function.

`decide` is your code. `grade` and the fixtures are supplied test infrastructure. The examples below specify expected answers independently; do not generate those answers with your function. An unimplemented starter is reported as NOT_SUBMITTED, never as a pass.


In [ ]:
def grade(candidate, cases):
    results = []
    for index, (case, expected) in enumerate(cases, 1):
        supplied = copy.deepcopy(case)
        try:
            observed = candidate(supplied)
        except NotImplementedError:
            results.append({"case": index, "status": "NOT_SUBMITTED"})
            continue
        except Exception as error:
            results.append({"case": index, "status": "FAILED", "error_type": type(error).__name__})
            continue
        try:
            passed = json.dumps(observed, sort_keys=True, allow_nan=False) == json.dumps(
                expected, sort_keys=True, allow_nan=False
            ) and json.dumps(supplied, sort_keys=True, allow_nan=False) == json.dumps(
                case, sort_keys=True, allow_nan=False
            )
        except (TypeError, ValueError):
            passed = False
        results.append(
            {
                "case": index,
                "status": "PASS" if passed else "FAILED",
                "expected": expected,
                "observed": observed,
            }
        )
    return results


def decide(case):
    # Replace this body with your implementation of the contract above.
    raise NotImplementedError("Write your function before consulting the worked solution.")

In [ ]:
CASES = [
    ({"due": 100, "now": 139, "interval": 10, "paused": False}, [1, 3, 140]),
    ({"due": 100, "now": 100, "interval": 10, "paused": False}, [1, 0, 110]),
    ({"due": 100, "now": 99, "interval": 10, "paused": False}, [0, 0, 100]),
    ({"due": 100, "now": 139, "interval": 10, "paused": True}, [0, 0, 100]),
]
submission_results = grade(decide, CASES)
print(json.dumps(submission_results, indent=2))

## 3. Inspect and run the cumulative reference (20 minutes)

checkpoints/ch07.py: main. Trace tick, scan, persisted generation, and the agent serve child without a prompt argument.

Open the named code before running it. Point to where an input reaches a decision and where that decision changes an observable result. The next cell executes the supplied chapter checkpoint; it is reference evidence, not a substitute for your implementation. Local supplier and worker processes use temporary state and are cleaned up by the checkpoint. Chapter 11 also launches a bounded local MCP process. Chapter 15 does not install a system service.


In [ ]:
# This supplied cumulative program is separate from grading your function.
# It uses fixture models/channels. Some chapters start local child processes.
# No --live, --telegram or --containers switch is added.
reference_environment = {
    k: v for k, v in os.environ.items() if k in {"PATH", "SYSTEMROOT", "TMPDIR", "LANG", "LC_ALL"}
}
reference_environment["PYTHONPATH"] = str(ROOT / "src")
reference_run = subprocess.run(
    [sys.executable, str(CHECKPOINT)],
    cwd=ROOT,
    env=reference_environment,
    capture_output=True,
    text=True,
    timeout=180,
    check=True,
)
print(reference_run.stdout)
EXPECTED_OBSERVATIONS = [
    "Coalesced missed runs: 3",
    "Unattended second episode: PASS",
    "Purchases: 0",
]
assert all(text in reference_run.stdout for text in EXPECTED_OBSERVATIONS)
print("REFERENCE_CHECKPOINT_PASSED — this is not your submission grade.")

## 4. Transfer the rule (20 minutes)

Feed the first result’s next_due back into the function with the same observed time. Then compare a schedule wake with the stock condition that clears at eight tubs and fires again when stock drops to one.

Add your new case to TRANSFER_CASES, with an independently calculated expected answer. A blank list means the transfer remains unsubmitted. Describe one limit of your function before comparing it with the runtime.


In [ ]:
TRANSFER_CASES = []  # Add (input, expected) pairs after writing your prediction.
transfer_results = grade(decide, TRANSFER_CASES)
print(json.dumps(transfer_results, indent=2) if transfer_results else "TRANSFER_NOT_SUBMITTED")

## 5. Worked solution — reveal after attempting the task

The following function is an answer key, not a replacement for your submission. Its case results are recorded separately. Your teacher grades your original function, explanation and transfer case.

The second pass at 139 returns [0,0,140]. Stock conditions use episodes and rearming, not the clock’s coalescing count. The real child process discovers the second episode from durable state, drafts seven vanilla tubs for 1750 pence and purchases nothing.


In [ ]:
def worked_decide(case):
    due, now, interval = case["due"], case["now"], case["interval"]
    if case["paused"] or now < due:
        return [0, 0, due]
    skipped = (now - due) // interval
    return [1, skipped, due + (skipped + 1) * interval]


worked_results = grade(worked_decide, CASES)
assert all(row["status"] == "PASS" for row in worked_results)
print("WORKED_EXAMPLE_PASSED; submission_results remains separate.")

## 6. Break the tempting implementation (10 minutes)

Explain why the following shortcut violates at least one case. Predict which case catches it before running. Then name a different defect the current cases might miss.


In [ ]:
def tempting_shortcut(case):
    return [1, 0, case["now"] + case["interval"]]


shortcut_results = grade(tempting_shortcut, CASES)
assert any(row["status"] == "FAILED" for row in shortcut_results)
print(json.dumps(shortcut_results, indent=2))

## Exit ticket (5 minutes)

Submit your prediction, original decide function, case results, one transfer case, and the runtime path you traced. Explain: (1) which boundary Python enforced, (2) what evidence came from the supplied program, and (3) what remains unproved.

**Misconception to resolve:** A heartbeat proving a process lives is not a work request. Advancing due time while paused silently loses work.

**Scope of this lab:** This arithmetic does not persist jobs, define time zones or prevent duplicate transactions. The runtime supplies those boundaries; the checkpoint proves a local child run, not long-term host availability.

A successful reference run or worked example does not establish learner mastery. Instructor guidance and answers are in the matching versioned guide.
